**只有数据损失**

In [ ]:
import torch
from torch import optim

import pinn_starlight_core.nn.Layers as Layers
import pinn_starlight_core.nn.Losses as Loss
import pinn_starlight_core.data.RAWLoader as RAWLoader
import pinn_starlight_core.data.FakeRAW as FakeRAW

raw_loader = RAWLoader.RAWLoader()
raw_loader.from_array(FakeRAW.FakeRaw().get_fake_raw())
coords, values = raw_loader.get_raw_data()

layers = [
    Layers.SkyglowLinear(2, 512),
    Layers.SkyglowActivation(),
    Layers.SkyglowLinear(512, 64),
    Layers.SkyglowActivation(),
    Layers.SkyglowLinear(64, 1),
]

params = []
for layer in layers:
    if isinstance(layer, Layers.SkyglowLinear):
        params += list(layer.parameters())

optimizer = optim.Adam(params, lr=0.001)
loss_fn = Loss.MSEData()

for epoch in range(5000):
    N = coords.shape[0]
    idx = torch.randint(0, N, size=(4096,))
    batch_xy = coords[idx]
    batch_I = values[idx]

    a = batch_xy
    for layer in layers:
        a = layer.forward(a)
    I_pred = a.squeeze()

    loss = loss_fn.forward(batch_I, I_pred)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Step {(epoch // 500) + 1}, Loss={loss.item():.6f}")


**包括物理损失**

出图

In [7]:
import torch
from torch import optim
from tqdm.notebook import tqdm

import pinn_starlight_core.nn.Layers as Layers
import pinn_starlight_core.nn.Losses as Loss
import pinn_starlight_core.data.RAWLoader as RAWLoader
import pinn_starlight_core.data.FakeRAW as FakeRAW

raw_loader = RAWLoader.RAWLoader()
raw_loader.from_array(FakeRAW.FakeRaw().get_fake_raw())
coords, values, W, H = raw_loader.get_raw_data()

layers = [
    Layers.SkyglowLinear(2, 512),
    Layers.SkyglowActivation(),
    Layers.SkyglowLinear(512, 64),
    Layers.SkyglowActivation(),
    Layers.SkyglowLinear(64, 1),
]

params = []
for layer in layers:
    if isinstance(layer, Layers.SkyglowLinear):
        params += list(layer.parameters())

optimizer = optim.Adam(params, lr=0.001)

ld = Loss.MSEData()
lp = Loss.MSEPhysics()

alpha = 9.0
bg = 0.3 * torch.cos(3.0 * coords[:, 0]) * torch.cos(3.0 * coords[:, 1])
I_city = (alpha - 18.0) * bg
phy_weight = 0.01

for step in tqdm(range((coords.shape[0] / 240).interger())):
    idx = torch.randint(0, coords.shape[0], (coords.shape[0] / 320,))
    batch_xy = coords[idx].clone().requires_grad_(True)
    batch_I = values[idx]

    a = batch_xy
    for layer in layers:
        a = layer.forward(a)
    I_pred = a.squeeze()

    data_loss = ld.forward(batch_I, I_pred)
    phys_loss = lp.forward(batch_I, I_pred, I_city[idx], alpha, phy_weight, batch_xy)
    loss = data_loss + phys_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f"Step {(step // 500) + 1}, data={data_loss.item() * 100:.6f}%, phys={phys_loss.item() * 100:.6f}%, total={loss.item() * 100:.6f}%")

  0%|          | 0/4000 [00:00<?, ?it/s]

Step 1, data=2.818248%, phys=1.887090%, total=4.705338%
Step 2, data=2.510181%, phys=1.707393%, total=4.217574%
Step 3, data=0.472882%, phys=0.159938%, total=0.632821%
Step 4, data=0.518996%, phys=0.051774%, total=0.570770%
Step 5, data=0.523352%, phys=0.026843%, total=0.550195%
Step 6, data=0.505804%, phys=0.018598%, total=0.524402%
Step 7, data=0.514517%, phys=0.010030%, total=0.524547%
Step 8, data=0.435716%, phys=0.010128%, total=0.445844%


In [2]:
import os
import torch
from torch import optim
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import pinn_starlight_core.nn.Layers as Layers
import pinn_starlight_core.nn.Losses as Loss
import pinn_starlight_core.data.RAWLoader as RAWLoader

input_dir  = '../../data/real_raw/origin'
output_dir = '../../data/real_raw/trained'
os.makedirs(output_dir, exist_ok=True)

for file in sorted(os.listdir(input_dir)):
    path = os.path.join(input_dir, file)
    base = file.rsplit('.', 1)[0]
    print(f'Processing {file}...')

    loader = RAWLoader.RAWLoader()
    loader.load(path)
    coords, values, W, H = loader.get_raw_data()

    layers = [
        Layers.SkyglowLinear(2, 512),
        Layers.SkyglowActivation(),
        Layers.SkyglowLinear(512, 64),
        Layers.SkyglowActivation(),
        Layers.SkyglowLinear(64, 1),
    ]

    params = []
    for layer in layers:
        if isinstance(layer, Layers.SkyglowLinear):
            params += list(layer.parameters())

    optimizer = optim.Adam(params, lr=0.001)
    ld = Loss.MSEData()
    lp = Loss.MSEPhysics()

    alpha = 9.0
    I_city = torch.ones(coords.shape[0]) * 0.5     # 真实数据用常数
    phy_weight = 0.01

    for step in tqdm(range((coords.shape[0] / 240).interger())):
        idx = torch.randint(0, coords.shape[0], (coords.shape[0] / 320,))
        batch_xy = coords[idx].clone().requires_grad_(True)
        batch_I = values[idx]

        a = batch_xy
        for layer in layers:
            a = layer.forward(a)
        I_pred = a.squeeze()

        data_loss = ld.forward(batch_I, I_pred)
        phys_loss = lp.forward(batch_I, I_pred, I_city[idx], alpha, phy_weight, batch_xy)
        loss = data_loss + phys_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        I_pred = torch.empty(coords.shape[0])
        for start in range(0, coords.shape[0], 50000):
            end = min(start + 50000, coords.shape[0])
            a = coords[start:end]
            for layer in layers:
                a = layer.forward(a)
            I_pred[start:end] = a.squeeze()

    obs = values.reshape(W, H).numpy()
    pred = I_pred.reshape(W, H).numpy()
    res = (obs - pred).clip(0, 1)

    plt.imsave(f'{output_dir}/{base}_observed.png', obs, cmap='gray')
    plt.imsave(f'{output_dir}/{base}_predicted.png', pred, cmap='gray')
    plt.imsave(f'{output_dir}/{base}_residual.png', res, cmap='gray')

print('Done.')

Processing 1.jpg...


  0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 